# Qwen-Image-2.1 Colab 破限与高频细节精修终极版笔记本 (8 大社区组件全集成 + 全 Bug 修复 + 原生 JPG 直出)

本笔记本在原版 **[Qwen-Image-2.1](https://huggingface.co/Qwen/Qwen-Image-2.1)** 部署流程基础上，完整集成了 **8 个开源社区高分破限底座、高频纹理 VAE、去审查编码器与细节修复 LoRA**，并修复了原版 4 项底层暗坑与 WebP 保存格式问题。

### 🧩 已集成的 8 大开源社区组件
| 编号 | 组件类别 | 社区模型 / 插件名称 | Hugging Face 仓库与权重文件 | 核心作用 |
| :---: | :---: | :--- | :--- | :--- |
| **1** | **微调底座** | **ChatR Labs Uncensored Omni (Step 90000 BF16 完整微调版)** | `chatrlabs/Qwen-Image-2.1-Uncensored-Omni-Quantized` (`qwen-image-2.1-step90000-BF16`) | 近 60 万张未过滤高质量图像微调 90,000 步（Rank-128 融合进 224 个模块），彻底补齐大尺度服装剪裁、人体解剖结构与姿态细节 |
| **2** | **高频 VAE** | **madebyollin Texture-Fix VAE** | `madebyollin/texture-fix-vae-for-qwen-image-2.1` | 知名作者 madebyollin 借助对抗损失微调的高频纹理修复 VAE，消除原版 VAE 解码时的涂抹模糊与棋盘格伪影 |
| **3** | **破限编码器** | **pottokao Text-Encoder-Heretic (Abliterated)** | `pottokao/Qwen-Image-2.1-Text-Encoder-Heretic` | 剥离 `Qwen3-VL-8B` 内部安全拒绝向量（Refusal Vector）并完整保留图生图 Vision Tower，防止敏感身体/服饰词在编码层被弱化 |
| **4** | **核心 LoRA 1** | **ChatR Labs Uncensored Omni LoRA v0.1** | `chatrlabs/Qwen-Image-2.1-Uncensored-Omni-LoRA-v0.1` (`chatrlabs-omni-lora-v0.1-bf16.safetensors`) | 独立可调权重的 Rank-128 破限与解剖插件（若底座选原版则默认 0.85；若底座已选 Step 90000 完整版则可按需微调叠加） |
| **5** | **核心 LoRA 2** | **e-n-v-y / Qwen-Image-2.1-Fix-v2.0** | `e-n-v-y/Qwen-Image-2.1-Fix-v2.0` (`qwen2.1-detail-fix-2.0.safetensors`) | 社区高分细节与解剖修复插件，大幅强化皮革、金属、布料织物纹理与手部/躯干结构精度 |
| **6** | **核心 LoRA 3** | **Elusarca's Detail Enhancer LoRA v1.0** | `reverentelusarca/elusarcas-qwen-2.1-detail-enhancer-lora` (`elusarcas-qwen2-1-detailer-v1.safetensors`) | 高频微观纹理增强器（自动注入触发词 `enhance this image, `），给美术出设定图时拉满服装接缝、配饰刻花与皮肤毛孔质感 |
| **7** | **核心 LoRA 4** | **ausboss Outfit-Swap & Consistency LoRA** | `ausboss/Qwen-Image-2.1-Outfit-Swap-Consistency-LoRA` (`qwen-image-2.1-outfit-swap.safetensors`) | 图生图换装/修改服装剪裁时，像素级锁定原图人脸五官与背景不发生偏移或变肉 |
| **8** | **核心 LoRA 5** | **prithivMLmods Natural Exposure LoRA** | `prithivMLmods/Qwen-Image-2.1-Natural-Exposure-LoRA` (`Qwen-Image-2.1-Natural-Exposure-LoRA-4000.safetensors`) | 自然曝光与真实皮肤漫反射修复插件，消除原版塑料油光感 |

---

### 🛠️ 已修复的 5 大核心问题
1. **修复 ComfyUI 融合层 `img_mlp.gate_up` 在 Diffusers 下加载报错**：内置 `convert_comfy_qwen21_lora_to_diffusers()` 转换器，自动将 ComfyUI 265 布局的 `.img_mlp.gate_up` (`[24576, r]`) 无损对半拆分为 Diffusers 297 布局的 `gate_layer` (`[12288, r]`) 与 `proj` (`[12288, r]`)，并精确折算逐层独立 `.alpha` 缩放（如 `e-n-v-y` 的 83/96）。
2. **修复 `true_cfg_scale` 因缺少 `negative_prompt` 而静默失效的问题**：`QwenImage21Pipeline` 源码中要求必须同时提供 `true_cfg_scale > 1` 与非空 `negative_prompt` 才会开启 CFG 引导。现已默认开启 `True CFG = 4.5` 并内置解剖/画质负面词（即使清空也会自动补单空格 `" "` 保持 CFG 激活）。
3. **修复 `vae.enable_tiling()` 导致的竖向色彩条纹伪影**：在 A100 (>=32GB) 上默认关闭分块解码，彻底消除高分辨率下的竖条纹接缝。
4. **修复 `output_resolution=1024` 强制压低参考图清晰度的问题**：新增 `output_resolution` 档位（默认 `1536`，可选 `2048`），防止 2K 参考图在进入 VAE 前被强制缩水。
5. **修复右键保存图片为 `WEBP` 格式的问题**：Gradio 4/5 默认将前端预览图转码为 `webp`。现已将 `gr.Image` 显式锁定为 `format="jpeg"`，且每次生成自动以 **4:4:4 无损色度抽样 + 98% 最高画质** 保存为 `.jpg` 文件——**无论直接在图片上右键「图片另存为」还是点击下方一键下载按钮，拿到的都是纯正高清 `.jpg` 格式！**

In [ ]:
#@title 1. 检查当前分配的 GPU 显存
!nvidia-smi

In [ ]:
#@title 2. 安全安装推理与多 LoRA 挂载依赖 (不改动 Colab 原生 PyTorch/CUDA 与 Pillow，免重启)
!pip install -q -U --upgrade-strategy only-if-needed "transformers>=5.17.0" accelerate "peft>=0.14.0" safetensors huggingface_hub gradio
!pip install -q --upgrade-strategy only-if-needed git+https://github.com/huggingface/diffusers.git
print("✅ 依赖安装完成！无需重启会话，请直接运行下一步。")

In [ ]:
#@title 3. 加载底座模型、Texture-Fix VAE 与 Heretic 去审查编码器
import gc
import torch
from diffusers import QwenImage21Pipeline, QwenImage21Transformer2DModel, AutoencoderKLQwenImage21
from transformers import Qwen3VLForConditionalGeneration

BASE_MODEL_ID = "Qwen/Qwen-Image-2.1"  #@param ["Qwen/Qwen-Image-2.1", "Qwen/Qwen-Image-2.1-Turbo"]
TRANSFORMER_SOURCE = "ChatR Labs Uncensored Omni (Step 90000 BF16 完整微调底座)"  #@param ["ChatR Labs Uncensored Omni (Step 90000 BF16 完整微调底座)", "Qwen/Qwen-Image-2.1 官方原版底座"]
USE_TEXTURE_FIX_VAE = True  #@param {type:"boolean"}
USE_HERETIC_TEXT_ENCODER = True  #@param {type:"boolean"}

def _load_pretrained_compat(cls, repo_id, dtype=torch.bfloat16, **kwargs):
    """兼容不同版本 transformers/diffusers 的 dtype 与 torch_dtype 参数名"""
    try:
        return cls.from_pretrained(repo_id, dtype=dtype, **kwargs)
    except TypeError:
        return cls.from_pretrained(repo_id, torch_dtype=dtype, **kwargs)

dtype = torch.bfloat16
pipe_kwargs = {}

# [组件 1] 加载 ChatR Labs Uncensored Omni (Step 90000 BF16 完整微调版 Transformer)
if "ChatR Labs" in TRANSFORMER_SOURCE and "Turbo" not in BASE_MODEL_ID:
    print("⏳ [1/3] 正在加载 ChatR Labs Uncensored Omni (Step 90000 BF16) 完整微调版 Transformer...")
    pipe_kwargs["transformer"] = _load_pretrained_compat(
        QwenImage21Transformer2DModel,
        "chatrlabs/Qwen-Image-2.1-Uncensored-Omni-Quantized",
        dtype=dtype,
        subfolder="qwen-image-2.1-step90000-BF16",
    )
else:
    print(f"⏳ [1/3] 使用 {BASE_MODEL_ID} 官方原版 Transformer...")

# [组件 2] 加载 madebyollin Texture-Fix VAE (消除高频皮肤/布料涂抹与棋盘格伪影)
if USE_TEXTURE_FIX_VAE:
    print("⏳ [2/3] 正在加载 madebyollin/texture-fix-vae-for-qwen-image-2.1 (高频纹理修复 VAE)...")
    pipe_kwargs["vae"] = _load_pretrained_compat(
        AutoencoderKLQwenImage21,
        "madebyollin/texture-fix-vae-for-qwen-image-2.1",
        dtype=dtype,
    )

# [组件 3] 加载 pottokao Text-Encoder-Heretic (剥离安全拒绝向量的 Qwen3-VL-8B 编码器)
if USE_HERETIC_TEXT_ENCODER:
    print("⏳ [3/3] 正在加载 pottokao/Qwen-Image-2.1-Text-Encoder-Heretic (去审查多模态编码器)...")
    pipe_kwargs["text_encoder"] = _load_pretrained_compat(
        Qwen3VLForConditionalGeneration,
        "pottokao/Qwen-Image-2.1-Text-Encoder-Heretic",
        dtype=dtype,
    )

pipe = _load_pretrained_compat(QwenImage21Pipeline, BASE_MODEL_ID, dtype=dtype, **pipe_kwargs)
MODEL_ID = BASE_MODEL_ID

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"🖥️ 检测到 GPU 显存: {vram_gb:.1f} GB")

# 修复原版 Bug：A100 显存充足时关闭 VAE Tiling，避免分块解码产生竖向色彩接缝条纹
if vram_gb >= 32:
    if hasattr(pipe, "vae") and hasattr(pipe.vae, "disable_tiling"):
        pipe.vae.disable_tiling()
    pipe.to("cuda")
    print("✅ 已将全套模型加载至 GPU (A100 无缝解码极速模式，已关闭 VAE Tiling 防条纹伪影)。")
else:
    if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
    pipe.enable_model_cpu_offload()
    print("✅ 显存小于 32GB (如 L4 24GB)，已启用 enable_model_cpu_offload() + VAE Tiling 防爆显存。")

gc.collect()
torch.cuda.empty_cache()

In [ ]:
#@title 3B. 装载全部 5 款社区破限与细节精修 LoRA 插件 (内置 ComfyUI 融合层 gate_up 自动无损拆分器)
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

if "pipe" not in globals():
    raise RuntimeError("请先运行上方【第 3 步】加载底座模型！")

# 若第 3 步已选用 ChatR Labs Step 90000 完整融合底座，则 Omni LoRA 默认权重设为 0.0（可在界面随时拉高叠加）；
# 若第 3 步选用官方原版底座，则 Omni LoRA 默认权重自动设为 0.85。
_default_omni_w = 0.0 if "ChatR Labs" in globals().get("TRANSFORMER_SOURCE", "") else 0.85

LORA_REGISTRY = {
    "omni": {
        "name": "① ChatR Labs Uncensored Omni LoRA v0.1 (Rank-128 破限与人体解剖)",
        "repo_id": "chatrlabs/Qwen-Image-2.1-Uncensored-Omni-LoRA-v0.1",
        "weight_name": "chatrlabs-omni-lora-v0.1-bf16.safetensors",
        "default_weight": _default_omni_w,
    },
    "envy_fix": {
        "name": "② e-n-v-y / Qwen-Image-2.1-Fix-v2.0 (解剖结构/手部躯干/皮革金属织物修复)",
        "repo_id": "e-n-v-y/Qwen-Image-2.1-Fix-v2.0",
        "weight_name": "qwen2.1-detail-fix-2.0.safetensors",
        "default_weight": 0.65,
    },
    "elusarca_detail": {
        "name": "③ Elusarca's Detail Enhancer v1.0 (服装接缝/配饰刻花/皮肤毛孔高频微观增强)",
        "repo_id": "reverentelusarca/elusarcas-qwen-2.1-detail-enhancer-lora",
        "weight_name": "elusarcas-qwen2-1-detailer-v1.safetensors",
        "default_weight": 0.55,
    },
    "outfit_swap": {
        "name": "④ ausboss Outfit-Swap Consistency LoRA (改衣/换装时锁定人脸与背景不偏移)",
        "repo_id": "ausboss/Qwen-Image-2.1-Outfit-Swap-Consistency-LoRA",
        "weight_name": "qwen-image-2.1-outfit-swap.safetensors",
        "default_weight": 0.35,
    },
    "natural_exposure": {
        "name": "⑤ prithivMLmods Natural Exposure LoRA (去除塑料高光/真实皮肤光影质感)",
        "repo_id": "prithivMLmods/Qwen-Image-2.1-Natural-Exposure-LoRA",
        "weight_name": "Qwen-Image-2.1-Natural-Exposure-LoRA-4000.safetensors",
        "default_weight": 0.35,
    },
}

PRELOAD_ALL_LORAS = True  #@param {type:"boolean"}

def convert_comfy_qwen21_lora_to_diffusers(raw_sd, target_dtype=torch.bfloat16):
    """
    修复 Diffusers 加载社区 Qwen-Image-2.1 LoRA 报错的核心转换函数：
    1. 统一剥离 'diffusion_model.' / 'transformer.' 前缀与 '.default.' 中缀；
    2. 自动解析逐层独立 '.alpha' 张量（如 e-n-v-y Fix v2.0 的 alpha=83.0/96.0）并均衡折算至 lora_A / lora_B；
    3. 将 ComfyUI 265 布局中的融合层 '.img_mlp.gate_up' ([24576, rank]) 沿第 0 维精确对半切分为
       Diffusers 297 布局的 '.img_mlp.gate_layer' ([12288, rank]) 与 '.img_mlp.proj' ([12288, rank])。
    """
    norm_sd = {}
    alphas = {}
    for k, v in raw_sd.items():
        if "dora_scale" in k:
            continue
        clean_k = k
        if clean_k.startswith("diffusion_model."):
            clean_k = clean_k[len("diffusion_model."):]
        elif clean_k.startswith("transformer."):
            clean_k = clean_k[len("transformer."):]
        clean_k = clean_k.replace(".default.", ".")
        clean_k = clean_k.replace(".lora_down.weight", ".lora_A.weight").replace(".lora_up.weight", ".lora_B.weight")
        if clean_k.endswith(".alpha"):
            alphas[clean_k[:-len(".alpha")]] = float(v.float().item())
        else:
            norm_sd[clean_k] = v

    base_modules = sorted({k[:-len(".lora_A.weight")] for k in norm_sd if k.endswith(".lora_A.weight")})
    out_sd = {}
    for base in base_modules:
        a_key = f"{base}.lora_A.weight"
        b_key = f"{base}.lora_B.weight"
        if a_key not in norm_sd or b_key not in norm_sd:
            continue
        lora_A = norm_sd[a_key].to(torch.float32)
        lora_B = norm_sd[b_key].to(torch.float32)
        rank = lora_A.shape[0]
        if base in alphas and rank > 0:
            scale = alphas[base] / float(rank)
            scale_down = scale
            scale_up = 1.0
            while scale_down * 2.0 < scale_up:
                scale_down *= 2.0
                scale_up /= 2.0
            lora_A = lora_A * scale_down
            lora_B = lora_B * scale_up

        lora_A = lora_A.to(target_dtype).contiguous()
        lora_B = lora_B.to(target_dtype).contiguous()

        if base.endswith(".img_mlp.gate_up"):
            prefix = base[:-len(".gate_up")]
            half = lora_B.shape[0] // 2
            out_sd[f"transformer.{prefix}.gate_layer.lora_A.weight"] = lora_A.clone()
            out_sd[f"transformer.{prefix}.gate_layer.lora_B.weight"] = lora_B[:half, :].contiguous()
            out_sd[f"transformer.{prefix}.proj.lora_A.weight"] = lora_A.clone()
            out_sd[f"transformer.{prefix}.proj.lora_B.weight"] = lora_B[half:, :].contiguous()
        else:
            out_sd[f"transformer.{base}.lora_A.weight"] = lora_A
            out_sd[f"transformer.{base}.lora_B.weight"] = lora_B
    return out_sd

LOADED_ADAPTERS = set()

def ensure_lora_loaded(adapter_key):
    if adapter_key in LOADED_ADAPTERS:
        return
    meta = LORA_REGISTRY[adapter_key]
    print(f"⬇️ 正在下载并转换 {meta['name']}...")
    local_path = hf_hub_download(repo_id=meta["repo_id"], filename=meta["weight_name"])
    raw_sd = load_file(local_path, device="cpu")
    clean_sd = convert_comfy_qwen21_lora_to_diffusers(raw_sd, target_dtype=torch.bfloat16)
    pipe.load_lora_weights(clean_sd, adapter_name=adapter_key)
    LOADED_ADAPTERS.add(adapter_key)
    print(f"✅ [{adapter_key}] 挂载成功！(共映射 {len(clean_sd)} 个 Diffusers PEFT 张量)")

def apply_lora_weights(weight_dict):
    active_names, active_weights = [], []
    for k, w in weight_dict.items():
        w = float(w)
        if abs(w) > 1e-4:
            ensure_lora_loaded(k)
            active_names.append(k)
            active_weights.append(w)
    if active_names:
        pipe.set_adapters(active_names, adapter_weights=active_weights)
        summary = ", ".join([f"{k}={w:.2f}" for k, w in zip(active_names, active_weights)])
        print(f"🔗 当前激活 LoRA 组合: {summary}")
    elif LOADED_ADAPTERS:
        pipe.disable_lora()
        print("⚪ 当前所有 LoRA 权重均为 0，已临时关闭 LoRA 适配器。")

if PRELOAD_ALL_LORAS:
    for k in LORA_REGISTRY:
        ensure_lora_loaded(k)

apply_lora_weights({k: v["default_weight"] for k, v in LORA_REGISTRY.items()})
print("🎉 全部社区增强组件装载完毕！可运行 4A 或 4B 开始生图。")

### 方式 A：命令行上传参考图并执行图生图（直出高清 `.jpg` 并自动触发浏览器下载，可选，也可直接跳到 4B）

In [ ]:
#@title 4A. 上传参考图并执行图生图推理 (输出 98% 最高画质 JPG 并自动下载)
import time
import torch
from google.colab import files
from PIL import Image, ImageOps

if "pipe" not in globals():
    raise RuntimeError("请先按顺序运行上面的第 2、3、3B 步！")

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("未上传图片！")
input_filename = next(iter(uploaded.keys()))
with Image.open(input_filename) as _im:
    input_image = ImageOps.exif_transpose(_im).convert("RGBA")

prompt = "保持人物面部与主体姿态不变，精细刻画服装剪裁接缝、皮革与金属配饰纹理、真实皮肤毛孔质感，电影级光影"  #@param {type:"string"}
negative_prompt = "blurry, indistinct, low quality, low resolution, washed-out colors, bad anatomy, deformed body, extra limbs, missing fingers, badly drawn hands, censored, mosaic, pixelation, plastic skin, airbrushed, overexposed"  #@param {type:"string"}
aspect_ratio = "3:4 (1792x2400)"  #@param ["Auto (跟随参考图比例)", "1:1 (2048x2048)", "9:16 (1536x2752)", "16:9 (2752x1536)", "3:4 (1792x2400)", "4:3 (2400x1792)"]
output_resolution = 1536  #@param [1024, 1536, 2048] {type:"raw"}
steps = 35  #@param {type:"slider", min:10, max:50, step:1}
true_cfg_scale = 4.5  #@param {type:"slider", min:1.0, max:8.0, step:0.5}
seed = 42  #@param {type:"integer"}
jpg_quality = 98  #@param {type:"slider", min:85, max:100, step:1}

ratio_map = {
    "Auto (跟随参考图比例)": None,
    "1:1 (2048x2048)": (2048, 2048),
    "9:16 (1536x2752)": (1536, 2752),
    "16:9 (2752x1536)": (2752, 1536),
    "3:4 (1792x2400)": (1792, 2400),
    "4:3 (2400x1792)": (2400, 1792),
}

final_prompt = prompt.strip()
if "enhance this image" not in final_prompt.lower():
    final_prompt = f"enhance this image, {final_prompt}"

kwargs = {
    "prompt": final_prompt,
    "negative_prompt": negative_prompt.strip() if negative_prompt.strip() else " ",
    "image": [input_image],
    "output_resolution": int(output_resolution),
    "use_kv_cache": True,
    "generator": torch.Generator("cpu").manual_seed(int(seed)),
}
if ratio_map[aspect_ratio] is not None:
    kwargs["width"], kwargs["height"] = ratio_map[aspect_ratio]
if "Turbo" not in MODEL_ID:
    kwargs["num_inference_steps"] = int(steps)
    kwargs["true_cfg_scale"] = float(true_cfg_scale)

with torch.inference_mode():
    result = pipe(**kwargs).images[0]

jpg_filename = f"qwen21_output_{int(time.time())}_seed{int(seed)}.jpg"
# 使用 4:4:4 无损色度抽样 (subsampling=0) 保存最高画质 JPG
result.convert("RGB").save(jpg_filename, format="JPEG", quality=int(jpg_quality), subsampling=0, optimize=True)
print(f"✅ 生成完成，已保存为高清 JPG: {jpg_filename}")
files.download(jpg_filename)
result

### 方式 B：启动 Gradio 终极版图形界面（推荐，支持 5 款 LoRA 实时混音、多图参考与右键直接保存 `.jpg`）
- **关于保存为 JPG**：已将前端图像组件的编码格式显式锁定为 `format="jpeg"`，并以 `4:4:4` 色度抽样 + `98%` 画质写入 `.jpg` 文件。**在右侧生成结果图上直接鼠标右键「图片另存为」，保存下来的就是 `.jpg` 格式（不再是 `.webp`）**；同时下方还提供了一键下载 `.jpg` 原图文件栏。

In [ ]:
#@title 4B. 启动 Gradio 图形工作台 (原生 JPG 预览/右键另存为 JPG + 8 大社区组件控制台)
import os
import time
import inspect
import torch
import gradio as gr
from PIL import Image, ImageOps

if "pipe" not in globals() or "LORA_REGISTRY" not in globals():
    raise RuntimeError("请先按顺序运行上面的第 2、3、3B 步加载底座与 LoRA 组件！")

OUTPUT_DIR = "qwen21_jpg_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RATIO_MAP = {
    "Auto (跟随参考图比例)": None,
    "1:1 (2048x2048 2K方图)": (2048, 2048),
    "9:16 (1536x2752 2K竖图)": (1536, 2752),
    "16:9 (2752x1536 2K横图)": (2752, 1536),
    "3:4 (1792x2400 2K设定图)": (1792, 2400),
    "4:3 (2400x1792 2K横版)": (2400, 1792),
    "2:3 (1696x2528 2K海报)": (1696, 2528),
    "3:2 (2528x1696 2K摄影)": (2528, 1696),
    "1:1 (1024x1024 标准1K)": (1024, 1024),
    "3:4 (896x1152 快速预览)": (896, 1152),
}

DEFAULT_NEG_PROMPT = (
    "blurry, indistinct, low quality, low resolution, washed-out colors, "
    "bad anatomy, deformed body, extra limbs, missing fingers, badly drawn hands, "
    "censored, mosaic, pixelation, plastic skin, airbrushed, overexposed"
)

def generate_edit(
    ref_files,
    prompt,
    neg_prompt,
    auto_elusarca_trigger,
    ratio_key,
    cond_res,
    steps,
    cfg_scale,
    seed,
    jpg_quality,
    w_omni,
    w_envy,
    w_elusarca,
    w_outfit,
    w_exposure,
):
    # 1. 动态更新 5 款社区 LoRA 权重组合
    apply_lora_weights({
        "omni": w_omni,
        "envy_fix": w_envy,
        "elusarca_detail": w_elusarca,
        "outfit_swap": w_outfit,
        "natural_exposure": w_exposure,
    })

    # 2. 若启用了 Elusarca Detail Enhancer，自动补齐官方触发词 'enhance this image, '
    final_prompt = (prompt or "").strip()
    if float(w_elusarca) > 1e-4 and auto_elusarca_trigger and "enhance this image" not in final_prompt.lower():
        final_prompt = f"enhance this image, {final_prompt}" if final_prompt else "enhance this image"

    # 3. 修复 Diffusers 暗坑：确保 negative_prompt 绝不为空（空字符串会直接关闭 True CFG）
    final_neg = neg_prompt.strip() if (neg_prompt and neg_prompt.strip()) else " "

    kwargs = {
        "prompt": final_prompt,
        "negative_prompt": final_neg,
        "output_resolution": int(cond_res),
        "use_kv_cache": True,
        "generator": torch.Generator("cpu").manual_seed(int(seed)),
    }

    if RATIO_MAP.get(ratio_key) is not None:
        kwargs["width"], kwargs["height"] = RATIO_MAP[ratio_key]

    if ref_files:
        pil_images = []
        for f in ref_files[:10]:
            path = f if isinstance(f, str) else f.name
            with Image.open(path) as img:
                pil_images.append(ImageOps.exif_transpose(img).convert("RGBA"))
        kwargs["image"] = pil_images

    if "Turbo" not in MODEL_ID:
        kwargs["num_inference_steps"] = int(steps)
        kwargs["true_cfg_scale"] = float(cfg_scale)

    with torch.inference_mode():
        out_pil = pipe(**kwargs).images[0]

    # 4. 保存为 4:4:4 无损色度抽样 (subsampling=0) 的最高画质 .jpg 文件
    ts = time.strftime("%Y%m%d_%H%M%S")
    jpg_path = os.path.join(OUTPUT_DIR, f"qwen21_{ts}_seed{int(seed)}.jpg")
    out_pil.convert("RGB").save(
        jpg_path,
        format="JPEG",
        quality=int(jpg_quality),
        subsampling=0,
        optimize=True,
    )
    return jpg_path, jpg_path

# 兼容检查 gr.Image 是否支持 format='jpeg' 参数（Gradio 4/5 默认 format='webp' 导致右键保存变成 WEBP）
_img_init_params = inspect.signature(gr.Image.__init__).parameters
_img_extra_kwargs = {"format": "jpeg"} if "format" in _img_init_params else {}

with gr.Blocks(title="Qwen-Image-2.1 终极破限与细节精修工作台") as demo:
    gr.Markdown(
        f"## 🎨 Qwen-Image-2.1 终极破限与高频细节图生图工作台\n"
        f"- **当前底座**: `{MODEL_ID}` | **Transformer**: `{globals().get('TRANSFORMER_SOURCE', '默认')}`\n"
        f"- **Texture-Fix VAE**: `{'✅ 已启用 (madebyollin)' if globals().get('USE_TEXTURE_FIX_VAE') else '原版'}` | "
        f"**Heretic 破限编码器**: `{'✅ 已启用 (pottokao)' if globals().get('USE_HERETIC_TEXT_ENCODER') else '原版'}`\n"
        f"- **图片保存格式**: **原生高清 `.jpg` (4:4:4 色度抽样)** —— 直接在右侧预览图上**右键 -> 图片另存为**即为 `.jpg` 格式！"
    )
    with gr.Row():
        with gr.Column(scale=5):
            ref_input = gr.File(
                label="上传参考图（支持上传 1~10 张多图参考，留空则为纯文生图）",
                file_count="multiple",
                file_types=["image"],
                type="filepath",
            )
            prompt_box = gr.Textbox(
                label="编辑 / 生成提示词 (Prompt)",
                lines=3,
                placeholder="例如：保持人物面部不变，精细刻画服装剪裁、皮革纹理与皮肤毛孔质感，35mm胶片光影",
            )
            neg_box = gr.Textbox(
                label="反向提示词 (Negative Prompt — 留空也会自动补空格以保证 True CFG 生效)",
                value=DEFAULT_NEG_PROMPT,
                lines=2,
            )
            with gr.Row():
                ratio_dd = gr.Dropdown(
                    choices=list(RATIO_MAP.keys()),
                    value="3:4 (1792x2400 2K设定图)",
                    label="输出分辨率与比例",
                )
                cond_res_dd = gr.Dropdown(
                    choices=[1024, 1536, 2048],
                    value=1536,
                    label="参考图编码分辨率 (output_resolution)",
                )
            with gr.Row():
                steps_slider = gr.Slider(minimum=10, maximum=50, value=35, step=1, label="推理步数 (推荐 35~40)")
                cfg_slider = gr.Slider(minimum=1.0, maximum=8.0, value=4.5, step=0.5, label="True CFG Scale (推荐 4.0~5.0)")
            with gr.Row():
                seed_num = gr.Number(value=42, label="随机种子 (Seed)", precision=0)
                jpg_quality_slider = gr.Slider(minimum=85, maximum=100, value=98, step=1, label="JPG 导出画质 (Quality)")

            with gr.Accordion("🧩 社区破限与细节精修 LoRA 混音台 (支持 5 款插件实时叠加调参)", open=True):
                w_omni = gr.Slider(
                    0.0, 1.2, value=LORA_REGISTRY["omni"]["default_weight"], step=0.05,
                    label=LORA_REGISTRY["omni"]["name"] + (" [注：当前底座已是 Step90000 完整版，默认 0 即可，需更强破限可拉高]" if LORA_REGISTRY["omni"]["default_weight"] == 0.0 else ""),
                )
                w_envy = gr.Slider(0.0, 1.2, value=LORA_REGISTRY["envy_fix"]["default_weight"], step=0.05, label=LORA_REGISTRY["envy_fix"]["name"])
                w_elusarca = gr.Slider(0.0, 1.2, value=LORA_REGISTRY["elusarca_detail"]["default_weight"], step=0.05, label=LORA_REGISTRY["elusarca_detail"]["name"])
                auto_trig = gr.Checkbox(value=True, label="启用 Elusarca 时自动前置触发词 'enhance this image, '")
                w_outfit = gr.Slider(0.0, 1.2, value=LORA_REGISTRY["outfit_swap"]["default_weight"], step=0.05, label=LORA_REGISTRY["outfit_swap"]["name"])
                w_exposure = gr.Slider(0.0, 1.2, value=LORA_REGISTRY["natural_exposure"]["default_weight"], step=0.05, label=LORA_REGISTRY["natural_exposure"]["name"])

            run_btn = gr.Button("🚀 开始生成 / 精修 (直出 JPG)", variant="primary")

        with gr.Column(scale=5):
            output_img = gr.Image(
                label="生成结果 (支持直接右键『图片另存为』保存为 .jpg)",
                type="filepath",
                **_img_extra_kwargs,
            )
            output_file = gr.File(label="📥 一键下载原图 (.jpg 文件)")

    run_btn.click(
        fn=generate_edit,
        inputs=[
            ref_input, prompt_box, neg_box, auto_trig,
            ratio_dd, cond_res_dd, steps_slider, cfg_slider, seed_num, jpg_quality_slider,
            w_omni, w_envy, w_elusarca, w_outfit, w_exposure,
        ],
        outputs=[output_img, output_file],
    )

demo.launch(share=True, debug=True)